# Day 024 Project: Web Page → Structured JSON

## What You're Building

A `PageExtractor` class that turns any URL into a validated Pydantic model instance — no CSS selectors, no fragile HTML parsing. Define your target schema, point it at a URL, get structured data back.

The same class works on any site: swap the Pydantic model for the data you want and point it at a different URL.

## Project Requirements

1. Implement `PageExtractor(model_class, model='llama3.2')` with:
   - `self.fields` from `model_class.model_fields.keys()` (Pydantic v2)
   - `extract(url)` → `model_class` instance or `None`
   - `extract_many(urls)` → list of result envelopes
   - `to_json(instance)` → pretty-printed JSON string
2. Define a `SiteInfo` Pydantic model with at least `site_name` and `main_purpose`
3. Extract from `https://books.toscrape.com` and print the result
4. Run batch over 2 URLs (one good, one bad) and show both statuses

**Deliverable:** Run the extractor, print the JSON output, confirm the bad URL shows `status='error'`.

In [ ]:
import re
import json
import requests
import ollama
from bs4 import BeautifulSoup
from pydantic import BaseModel

## Provided: All Helper Functions

In [ ]:
def clean_html_text(html_string: str) -> str:
    soup = BeautifulSoup(html_string, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_schema_fields(
    text: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    fields_json = json.dumps(fields)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a structured data extractor. "
                    f"Extract the following fields from the text: {fields_json}. "
                    "Return JSON with exactly these keys. "
                    "Use null for any field you cannot find. "
                    "Return only valid JSON, no explanation."
                ),
            },
            {
                "role": "user",
                "content": f"Extract from this text:\n\n{text[:3000]}",
            },
        ],
        format="json",
    )
    raw = response["message"]["content"]
    try:
        return json.loads(raw)
    except Exception:
        return {f: None for f in fields}


def validate_extracted(raw: dict, model_class: type[BaseModel]) -> BaseModel | None:
    try:
        return model_class.model_validate(raw)
    except Exception:
        return None


def scrape_and_extract(
    url: str, fields: list[str], model: str = "llama3.2"
) -> dict:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    text = clean_html_text(response.text)
    return extract_schema_fields(text, fields, model=model)


def batch_scrape_extract(
    urls: list[str],
    fields: list[str],
    model: str = "llama3.2",
) -> list[dict]:
    results = []
    for url in urls:
        try:
            data = scrape_and_extract(url, fields, model=model)
            results.append({"url": url, "status": "ok", "data": data})
        except Exception as e:
            results.append({"url": url, "status": "error", "error": str(e)})
    return results

## Step 1: Define Your Schema

Define a Pydantic model for the data you want to extract. Use `str | None = None` for all fields — this makes validation always succeed even when the LLM leaves some fields blank.

In [ ]:
class SiteInfo(BaseModel):
    site_name: str | None = None
    main_purpose: str | None = None
    # TODO: add more fields if you like, e.g. language, num_products
    pass

## Step 2: Implement PageExtractor

Use `model_class.model_fields.keys()` (Pydantic v2) to get the field list. Delegate to helper functions you built in exercises 1-5.

In [ ]:
class PageExtractor:
    def __init__(self, model_class: type[BaseModel], model: str = 'llama3.2'):
        self.model_class = model_class
        self.llm_model = model
        # TODO: self.fields = list(model_class.model_fields.keys())
        pass

    def extract(self, url: str) -> BaseModel | None:
        # TODO: data = scrape_and_extract(url, self.fields, model=self.llm_model)
        # TODO: return validate_extracted(data, self.model_class)
        pass

    def extract_many(self, urls: list[str]) -> list[dict]:
        # TODO: return batch_scrape_extract(urls, self.fields, model=self.llm_model)
        pass

    def to_json(self, instance: BaseModel) -> str:
        # TODO: return json.dumps(instance.model_dump(), indent=2)
        pass

## Step 3: Use Your Extractor

In [ ]:
# extractor = PageExtractor(SiteInfo)
# info = extractor.extract("https://books.toscrape.com")
# if info:
#     print(extractor.to_json(info))
# else:
#     print("Extraction returned None (some required fields missing)")


In [ ]:
# results = extractor.extract_many(["https://books.toscrape.com", "http://localhost:9999/"])
# for r in results:
#     print(f"{r['url'][:45]} → {r['status']}")


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: PageExtractor class with required methods
    try:
        assert 'PageExtractor' in globals(), 'PageExtractor not defined'
        for m in ('extract', 'extract_many', 'to_json'):
            assert hasattr(PageExtractor, m), f'PageExtractor missing: {m}'
        passed += 1; print('\u2705 Check 1: PageExtractor has all required methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: extractor is a PageExtractor
    try:
        assert 'extractor' in globals(), 'extractor not defined'
        assert isinstance(extractor, PageExtractor), \
            f'extractor must be PageExtractor, got {type(extractor)}'
        passed += 1; print('\u2705 Check 2: extractor is a PageExtractor')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: extractor.fields is a non-empty list
    try:
        assert 'extractor' in globals(), 'extractor not defined'
        assert hasattr(extractor, 'fields'), 'extractor missing .fields'
        assert isinstance(extractor.fields, list) and len(extractor.fields) >= 1, \
            f'extractor.fields should be non-empty list: {extractor.fields!r}'
        passed += 1; print(f'\u2705 Check 3: extractor.fields = {extractor.fields}')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: results is a list with 2 items
    try:
        assert 'results' in globals(), 'results not defined'
        assert isinstance(results, list) and len(results) == 2, \
            f'results must have 2 items, got {results!r}'
        statuses = [r.get('status') for r in results]
        assert 'ok' in statuses, f"no 'ok' status in results: {statuses}"
        assert 'error' in statuses, f"no 'error' status in results: {statuses}"
        passed += 1; print('\u2705 Check 4: results has ok + error statuses')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: to_json returns a valid JSON string
    try:
        dummy = SiteInfo(site_name='Test', main_purpose='Testing')
        json_str = extractor.to_json(dummy)
        assert isinstance(json_str, str), \
            f'to_json should return str, got {type(json_str)}'
        parsed = json.loads(json_str)
        assert parsed.get('site_name') == 'Test', \
            f"json has wrong site_name: {parsed}"
        passed += 1; print('\u2705 Check 5: to_json returns valid indented JSON')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add a `BookInfo(BaseModel)` with `title`, `price`, `rating`, `description` fields (all `str | None`) and run `PageExtractor(BookInfo)` on a specific book detail page
- Add a `save_json(results, path)` method that writes batch results to a JSON file
- Add a `retry(url, fields, attempts=3)` method that retries extraction until validate_extracted returns a non-None result
- Compare: selector-based extraction (Day 23) vs AI extraction (Day 24) — which is more accurate for book titles? Which is faster?
- Extend to extract from multiple category pages of books.toscrape.com and aggregate the results